## Parse Structured XML Generated by Grobid

In [ ]:
import json
import logging
import Levenshtein
import os
import requests

from bs4 import BeautifulSoup

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
TEI_XML_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/tei/'
TEI_XML_CROSSREF_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/tei_crossref_citations/'
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_full/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_grobid/'

LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD = 5

In [ ]:
def extract_grouped_refs(soup):
    """
    Extract references grouped by occurrence in the same paragraph of the paper.
    """
    text = soup.find('text')
    grouped_refs = []
    for child_tag in text.body.children:
        # Skip empty lines and non-text tags (e.g., notes)
        if child_tag.name != 'div':
            continue

        # Extract title if present
        title = child_tag.head.text if child_tag.head else ''

        # Group all references from a single div
        ref_tags = child_tag.find_all('ref', attrs={"type": "bibr"})
        refs = [(ref_tag.get('target', None), ref_tag.text)
                for ref_tag in ref_tags]
        grouped_refs.append({'title': title, 'references': refs})
    return grouped_refs

In [ ]:
def extract_refs(soup):
    """
    Extract list of references for the paper.
    """
    bibliography = soup.find('listBibl')
    refs = {}
    for bibl_struct in bibliography.children:
        # Skip empty lines
        if bibl_struct.name != 'biblStruct':
            continue

        ref_id = bibl_struct['xml:id']
        ref_text = ''
        if bibl_struct.analytic and bibl_struct.analytic.title:
            ref_text = bibl_struct.analytic.title.text
        refs[ref_id] = ref_text
    return refs

In [ ]:
def parse_xml(filename):
    with open(filename, 'r') as xml_file:
        soup = BeautifulSoup(xml_file, 'xml')
    
    grouped_refs = extract_grouped_refs(soup)
    refs = extract_refs(soup)
    return grouped_refs, refs

In [ ]:
def export_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

In [ ]:
def process_xml(filename):
    pmid = filename.split('.')[0]
    xml_raw = os.path.join(TEI_XML_FOLDER, filename)
    xml_crossref = os.path.join(TEI_XML_CROSSREF_FOLDER, filename)

    grouped_refs_raw, refs_raw = parse_xml(xml_raw)
    grouped_refs_crossref, refs_crossref = parse_xml(xml_crossref)
    
    if grouped_refs_raw != grouped_refs_crossref:
        print('Grouped refs raw vs crossref differ')
        
    refs_merged = {}
    for ref_id, ref_raw in refs_raw.items():
        ref_fixed = refs_crossref[ref_id]
        distance = Levenshtein.distance(ref_raw.lower(), ref_fixed.lower())
        
        selected = False
        reason = ''
        if distance > LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD and ref_raw:
            selected = True
            reason = ref_raw
        merged_ref = {
            'title': ref_fixed,
            'selected': selected,
            'reason': reason
        }

        refs_merged[ref_id] = merged_ref
    
    grouped_refs_filename = os.path.join(GROUPED_REFS_FOLDER, f'{pmid}.json')
    export_to_json(grouped_refs_raw, grouped_refs_filename)
        
    refs_filename = os.path.join(REFS_FOLDER, f'{pmid}.json')
    # Skip merging
    export_to_json(refs_raw, refs_filename)

In [ ]:
for folder in [GROUPED_REFS_FOLDER, REFS_FOLDER]:
    if not os.path.exists(folder):
        os.mkdir(folder)
for filename in os.listdir(TEI_XML_FOLDER):
    if filename.endswith('.xml'):
        logging.info(f'Processing {filename}')
        process_xml(filename)

## Validating Grouped References

In [1]:
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_validated/'

### 1. JSON is valid - OK

In [ ]:
import json
import os

for file in sorted(os.listdir(GROUPED_REFS_FOLDER)):
    print(file, end='\t')
    filename = os.path.join(GROUPED_REFS_FOLDER, file)
    with open(filename, 'r') as f:
        data = json.load(f)
    print('OK')

## Obtaining the Clustering with PMIDs

In [34]:
import gzip
import json
import logging
import os

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig
from pysrc.papers.db.loaders import Loaders

In [35]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [36]:
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_validated/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_grobid/'
PUBTRENDS_EXPORT_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/pubtrends_export/'
CLUSTERING_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/clustering/'

In [37]:
PUBTRENDS_CONFIG = PubtrendsConfig(test=False)


def reload_exported_analyzer(path_to_archive, source='Pubmed'):
    """
    Load analysis data from json.gz archive generated by PubTrends.
    """
    with gzip.open(path_to_archive, 'rt', encoding='UTF-8') as zipfile:
        data = json.load(zipfile)

    loader, url_prefix = Loaders.get_loader_and_url_prefix(source, PUBTRENDS_CONFIG)
    analyzer = KeyPaperAnalyzer(loader, PUBTRENDS_CONFIG)
    analyzer.init(data)
    
    return analyzer

In [38]:
def reference_index(analyzer, review_pmid):
    reference_pmids = list(analyzer.citations_graph.successors(review_pmid))
    pubmed_data = analyzer.df[analyzer.df['id'].isin(reference_pmids)]
    pubmed_ref_search_index = {v.lower(): k for k, v in zip(pubmed_data['id'], pubmed_data['title'])}
    return pubmed_ref_search_index

In [39]:
def grobid2pmid(refs, reference_index):
    mapping = {}

    for key, ref in refs.items():
        if ref.lower() in reference_index:
            pmid = reference_index[ref.lower()]
            mapping[key] = int(pmid)
            
    return mapping

In [40]:
def export_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

In [41]:
def pmid_clustering(grouped_refs, pmid_mapping, prefix=''):
    clustering = []
    for el in grouped_refs:
        if isinstance(el, dict):
            new_element = {
                'title': el['title'],
                'references': pmid_clustering(el['references'], pmid_mapping, prefix='> ')
            }
        else:
            new_element = None
            if el[0]:  # Some references do not have IDs due to parsing error
                grobid_id = el[0][1:]  # '#b0' -> 'b0'
                if grobid_id in pmid_mapping:
                    new_element = pmid_mapping[grobid_id]
    
        if new_element:
            clustering.append(new_element)
            
    return clustering

In [42]:
def obtain_clustering(file):
    pmid, ext = os.path.splitext(file)
    
    logging.info(f'{pmid}: loading file with grouped references')
    filename = os.path.join(GROUPED_REFS_FOLDER, file)
    with open(filename, 'r') as f:
        data = json.load(f)

    logging.info(f'{pmid}: loading file with PubTrends analyzer')
    analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
    analyzer = reload_exported_analyzer(analysis_file)
    
    logging.info(f'{pmid}: building reference index for matching titles and PMIDs')
    ref_index = reference_index(analyzer, pmid)
    
    logging.info(f'{pmid}: loading file with references')
    refs_file = os.path.join(REFS_FOLDER, file)
    with open(refs_file, 'r') as f:
        refs = json.load(f)
        
    logging.info(f'{pmid}: building mapping between Grobid reference IDs and PMIDs')
    mapping = grobid2pmid(refs, ref_index)
    print(f'{pmid}: {len(mapping)} / {analyzer.citations_graph.out_degree(pmid)} references mapped')
    
    logging.info(f'{pmid}: building clustering with PMIDs instead of Grobid IDs')
    clustering = pmid_clustering(data, mapping)
    clustering_file = os.path.join(CLUSTERING_FOLDER, file)
    
    logging.info(f'{pmid}: exporting clustering')
    export_to_json(clustering, clustering_file)
    
    logging.info(f'{pmid}: done\n')
    return len(mapping), analyzer.citations_graph.out_degree(pmid)

In [44]:
if not os.path.exists(CLUSTERING_FOLDER):
    os.mkdir(CLUSTERING_FOLDER)

refs_mapped = 0
refs_total = 0
for file in sorted(os.listdir(GROUPED_REFS_FOLDER)):
    new_refs_mapped, new_refs_total = obtain_clustering(file)
    refs_mapped += new_refs_mapped
    refs_total += new_refs_total

2021-02-06 20:30:54,192 INFO: 26580716: loading file with grouped references
2021-02-06 20:30:54,194 INFO: 26580716: loading file with PubTrends analyzer
2021-02-06 20:30:54,863 INFO: 26580716: building reference index for matching titles and PMIDs
2021-02-06 20:30:54,867 INFO: 26580716: loading file with references
2021-02-06 20:30:54,868 INFO: 26580716: building mapping between Grobid reference IDs and PMIDs
2021-02-06 20:30:54,870 INFO: 26580716: building clustering with PMIDs instead of Grobid IDs
2021-02-06 20:30:54,871 INFO: 26580716: exporting clustering
2021-02-06 20:30:54,880 INFO: 26580716: done

2021-02-06 20:30:54,920 INFO: 26580717: loading file with grouped references
2021-02-06 20:30:54,922 INFO: 26580717: loading file with PubTrends analyzer


26580716: 96 / 152 references mapped


2021-02-06 20:30:55,894 INFO: 26580717: building reference index for matching titles and PMIDs
2021-02-06 20:30:55,896 INFO: 26580717: loading file with references
2021-02-06 20:30:55,900 INFO: 26580717: building mapping between Grobid reference IDs and PMIDs
2021-02-06 20:30:55,902 INFO: 26580717: building clustering with PMIDs instead of Grobid IDs
2021-02-06 20:30:55,903 INFO: 26580717: exporting clustering
2021-02-06 20:30:55,911 INFO: 26580717: done

2021-02-06 20:30:55,962 INFO: 26656254: loading file with grouped references
2021-02-06 20:30:55,965 INFO: 26656254: loading file with PubTrends analyzer


26580717: 78 / 91 references mapped


2021-02-06 20:30:56,735 INFO: 26656254: building reference index for matching titles and PMIDs
2021-02-06 20:30:56,739 INFO: 26656254: loading file with references
2021-02-06 20:30:56,742 INFO: 26656254: building mapping between Grobid reference IDs and PMIDs
2021-02-06 20:30:56,744 INFO: 26656254: building clustering with PMIDs instead of Grobid IDs
2021-02-06 20:30:56,745 INFO: 26656254: exporting clustering
2021-02-06 20:30:56,748 INFO: 26656254: done

2021-02-06 20:30:56,778 INFO: 26667849: loading file with grouped references
2021-02-06 20:30:56,782 INFO: 26667849: loading file with PubTrends analyzer


...

32020081: 163 / 178 references mapped


2021-02-06 20:31:30,103 INFO: 32042144: building reference index for matching titles and PMIDs
2021-02-06 20:31:30,106 INFO: 32042144: loading file with references
2021-02-06 20:31:30,108 INFO: 32042144: building mapping between Grobid reference IDs and PMIDs
2021-02-06 20:31:30,110 INFO: 32042144: building clustering with PMIDs instead of Grobid IDs
2021-02-06 20:31:30,113 INFO: 32042144: exporting clustering
2021-02-06 20:31:30,120 INFO: 32042144: done

2021-02-06 20:31:30,167 INFO: 32699292: loading file with grouped references
2021-02-06 20:31:30,169 INFO: 32699292: loading file with PubTrends analyzer


32042144: 134 / 204 references mapped


2021-02-06 20:31:30,846 INFO: 32699292: building reference index for matching titles and PMIDs
2021-02-06 20:31:30,848 INFO: 32699292: loading file with references
2021-02-06 20:31:30,850 INFO: 32699292: building mapping between Grobid reference IDs and PMIDs
2021-02-06 20:31:30,851 INFO: 32699292: building clustering with PMIDs instead of Grobid IDs
2021-02-06 20:31:30,852 INFO: 32699292: exporting clustering
2021-02-06 20:31:30,855 INFO: 32699292: done



32699292: 136 / 153 references mapped


In [45]:
refs_mapped / refs_total

0.7878469617404351

## Bulk PubTrends Export

In [ ]:
import gzip
import json
import logging
import os

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig, DEFAULT_ANALYZER_SETTINGS
from pysrc.papers.db.loaders import Loaders
from pysrc.papers.db.search_error import SearchError
from pysrc.papers.plot.plotter import visualize_analysis
from pysrc.papers.progress import Progress
from pysrc.papers.utils import SORT_MOST_CITED, ZOOM_OUT, PAPER_ANALYSIS

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
TARGET_FOLDER = '/home/willenjoy/work/pubtrends/local/pubtrends_export/'

TARGET_PMIDS = [26667849, 26678314, 26688349, 26688350, 26580716, 26580717, 26656254, 26675821, 27834397, 27834398,
                27890914, 27916977, 27677859, 27677860, 27904142, 28003656, 29147025, 29170536, 28853444, 28920587,
                28792006, 28852220, 29213134, 29321682, 30578414, 30842595, 30644449, 30679807, 30108335, 30390028,
                30459365, 30467385, 31686003, 31806885, 31836872, 32005979, 31937935, 32020081, 32042144, 32699292]

SOURCE = 'Pubmed'
LIMIT = 500

In [ ]:
def export_analysis(pmid):
    logging.info(f'Started analysis for PMID {pmid}')
    
    ids = [pmid]
    query = f'Paper ID: {pmid}'
    
    # extracted from 'analyze_id_list' Celery task
    config = PubtrendsConfig(test=False)
    loader = Loaders.get_loader(SOURCE, config)
    analyzer = KeyPaperAnalyzer(loader, config)
    settings = DEFAULT_ANALYZER_SETTINGS
    try:
        # Fetch references at first
        ids = ids + analyzer.load_references(
            ids[0], limit=LIMIT
        )
        # And then expand
        ids = analyzer.expand_ids(
            ids, limit=LIMIT,
            expand_steps=settings.EXPAND_STEPS, expand_citations_sigma=settings.EXPAND_CITATIONS_SIGMA,
            expand_similarity_threshold=settings.EXPAND_SIMILARITY_THRESHOLD,
            current=1, task=None
        )

        analyzer.analyze_papers(ids, query, task=None)
    finally:
        loader.close_connection()

    dump = analyzer.dump()
    
    # export as JSON
    path = os.path.join(TARGET_FOLDER, f'{pmid}.json.gz')
    with gzip.open(path, 'w') as f:
        f.write(json.dumps(dump).encode('utf-8'))
    
    logging.info(f'Finished analysis for PMID {pmid}\n')

In [ ]:
TARGET_PMIDS = [49, 58, 59, 64]

In [ ]:
for pmid in TARGET_PMIDS:
    export_analysis(pmid)